# 01 — Generate Facial-Region Masks

This notebook creates initial segmentation masks for the CNN training dataset.

It takes images from:

```text
rppg_dataset/images/
```

and creates masks in:

```text
rppg_dataset/masks/
```

Classes:

- `0` = background
- `1` = forehead
- `2` = left cheek
- `3` = right cheek
- `4` = lower lip

The masks are generated from **MediaPipe Face Mesh landmarks**.

> These are **initial/pseudo-label masks**. Before using a large dataset for serious training, inspect and manually correct a representative subset because automatically generated regions are not perfect.


## 1. Install dependencies

In [ ]:
%pip install mediapipe opencv-python numpy pillow matplotlib tqdm

## 2. Imports and paths

In [ ]:
from pathlib import Path
import cv2
import numpy as np
import matplotlib.pyplot as plt
from PIL import Image
from tqdm.auto import tqdm
import mediapipe as mp

# Project folders
DATASET_DIR = Path("rppg_dataset")
IMAGE_DIR = DATASET_DIR / "images"
MASK_DIR = DATASET_DIR / "masks"

MASK_DIR.mkdir(parents=True, exist_ok=True)

print("Images:", IMAGE_DIR.resolve())
print("Masks:", MASK_DIR.resolve())

if not IMAGE_DIR.exists():
    raise FileNotFoundError(
        f"Could not find {IMAGE_DIR}. "
        "Make sure this notebook is inside your rPPG_Project folder."
    )


## 3. Find your images

Supported formats: JPG, JPEG, PNG.

Put your face images inside `rppg_dataset/images/`.


In [ ]:
image_files = sorted([
    p for p in IMAGE_DIR.iterdir()
    if p.suffix.lower() in [".jpg", ".jpeg", ".png"]
])

print(f"Found {len(image_files)} image(s).")

for p in image_files[:20]:
    print(" -", p.name)

if not image_files:
    raise RuntimeError(
        "No images found. Put at least one JPG/PNG image in rppg_dataset/images/."
    )


## 4. Initialize MediaPipe Face Mesh

In [ ]:
mp_face_mesh = mp.solutions.face_mesh

face_mesh = mp_face_mesh.FaceMesh(
    static_image_mode=True,
    max_num_faces=1,
    refine_landmarks=True,
    min_detection_confidence=0.5
)

print("MediaPipe initialized.")


## 5. Facial landmark helpers

We use facial landmarks to construct approximate regions that move with the person's face.

The regions are deliberately kept away from eyes, nostrils, and the mouth opening as much as possible because those areas are less useful for the rPPG signal.


In [ ]:
def landmark_xy(landmarks, index, width, height):
    x = int(np.clip(landmarks[index].x * width, 0, width - 1))
    y = int(np.clip(landmarks[index].y * height, 0, height - 1))
    return [x, y]


def polygon_from_landmarks(landmarks, indices, width, height):
    points = np.array(
        [landmark_xy(landmarks, i, width, height) for i in indices],
        dtype=np.int32
    )
    return points


## 6. Define the facial regions

The landmark indices below define approximate polygons for the forehead, cheeks, and lower-lip/chin region.

Because face shape varies, these are **starting labels**, not guaranteed perfect ground truth.


In [ ]:
# MediaPipe Face Mesh landmark indices.
# These polygons are intentionally approximate and conservative.

# Forehead: region above the eyebrows and inside the sides of the face.
FOREHEAD = [
    10, 338, 297, 332, 284, 251,
    389, 356, 454, 323, 361,
    9, 151, 108, 69, 104, 67, 103,
    54, 21, 162, 127, 234, 93
]

# Left cheek (person's left in image coordinates is not necessarily
# the viewer's left depending on mirroring; the labels are kept
# consistent with the image coordinate system).
LEFT_CHEEK = [
    50, 101, 118, 117, 111, 123,
    147, 192, 213, 177, 137,
    123, 50
]

# Right cheek.
RIGHT_CHEEK = [
    280, 330, 347, 346, 340, 352,
    376, 416, 433, 401, 366,
    352, 280
]

# Lower lip / lower-mouth region.
# We include a small surrounding lower-lip/chin area while avoiding
# the mouth opening as much as possible.
LOWER_LIP = [
    17, 314, 405, 320, 307, 375,
    321, 308, 324, 318, 402,
    317, 14, 87, 178, 88, 95, 78
]

print("Region definitions loaded.")


## 7. Generate one mask

Mask values:

```text
0 = background
1 = forehead
2 = left cheek
3 = right cheek
4 = lower lip
```


In [ ]:
def generate_mask(image_bgr):
    height, width = image_bgr.shape[:2]

    image_rgb = cv2.cvtColor(image_bgr, cv2.COLOR_BGR2RGB)
    result = face_mesh.process(image_rgb)

    mask = np.zeros((height, width), dtype=np.uint8)

    if not result.multi_face_landmarks:
        return mask, False, None

    landmarks = result.multi_face_landmarks[0].landmark

    forehead_poly = polygon_from_landmarks(
        landmarks, FOREHEAD, width, height
    )
    left_cheek_poly = polygon_from_landmarks(
        landmarks, LEFT_CHEEK, width, height
    )
    right_cheek_poly = polygon_from_landmarks(
        landmarks, RIGHT_CHEEK, width, height
    )
    lower_lip_poly = polygon_from_landmarks(
        landmarks, LOWER_LIP, width, height
    )

    # Draw each class into the mask.
    cv2.fillPoly(mask, [forehead_poly], 1)
    cv2.fillPoly(mask, [left_cheek_poly], 2)
    cv2.fillPoly(mask, [right_cheek_poly], 3)
    cv2.fillPoly(mask, [lower_lip_poly], 4)

    polygons = {
        "forehead": forehead_poly,
        "left_cheek": left_cheek_poly,
        "right_cheek": right_cheek_poly,
        "lower_lip": lower_lip_poly
    }

    return mask, True, polygons


## 8. Test the generator on one image

Before generating hundreds or thousands of masks, inspect one image.


In [ ]:
test_path = image_files[0]

test_bgr = cv2.imread(str(test_path))

if test_bgr is None:
    raise RuntimeError(f"Could not read {test_path}")

test_mask, found, polygons = generate_mask(test_bgr)

if not found:
    raise RuntimeError("No face was detected in the test image.")

test_rgb = cv2.cvtColor(test_bgr, cv2.COLOR_BGR2RGB)

print("Image:", test_path.name)
print("Mask classes:", np.unique(test_mask))

plt.figure(figsize=(14, 6))

plt.subplot(1, 2, 1)
plt.imshow(test_rgb)
plt.title("Original image")
plt.axis("off")

plt.subplot(1, 2, 2)
plt.imshow(test_mask, interpolation="nearest", vmin=0, vmax=4)
plt.title("Generated facial-region mask")
plt.colorbar(ticks=[0, 1, 2, 3, 4])
plt.axis("off")

plt.tight_layout()
plt.show()


## 9. Overlay the regions on the original image

This makes it easier to determine whether the generated labels are reasonable.


In [ ]:
def create_overlay(image_rgb, mask, alpha=0.45):
    # Visualization colors only; the saved mask still contains class IDs.
    overlay = np.zeros_like(image_rgb)

    overlay[mask == 1] = [255, 0, 0]      # forehead
    overlay[mask == 2] = [0, 255, 0]      # left cheek
    overlay[mask == 3] = [0, 0, 255]      # right cheek
    overlay[mask == 4] = [255, 255, 0]    # lower lip

    result = image_rgb.copy()
    active = mask != 0

    result[active] = (
        alpha * overlay[active] +
        (1 - alpha) * image_rgb[active]
    ).astype(np.uint8)

    return result


overlay = create_overlay(test_rgb, test_mask)

plt.figure(figsize=(10, 7))
plt.imshow(overlay)
plt.title("Generated regions")
plt.axis("off")
plt.show()


## 10. Generate masks for every image

Each mask is saved as a grayscale PNG with the same filename stem as its image.

Example:

```text
images/myface.jpg
        ↓
masks/myface.png
```


In [ ]:
successful = 0
failed = 0

for image_path in tqdm(image_files, desc="Generating masks"):
    image_bgr = cv2.imread(str(image_path))

    if image_bgr is None:
        failed += 1
        continue

    mask, found, _ = generate_mask(image_bgr)

    if not found:
        failed += 1
        continue

    mask_path = MASK_DIR / f"{image_path.stem}.png"

    # Save as single-channel PNG.
    cv2.imwrite(str(mask_path), mask)

    successful += 1

print()
print("Successfully generated:", successful)
print("Failed/no face:", failed)
print("Masks saved to:", MASK_DIR.resolve())


## 11. Check the generated dataset

This displays several random examples so you can inspect the pseudo-labels.


In [ ]:
mask_files = sorted(MASK_DIR.glob("*.png"))

print("Masks created:", len(mask_files))

if mask_files:
    n = min(6, len(mask_files))
    selected = mask_files[:n]

    fig, axes = plt.subplots(n, 2, figsize=(10, 4 * n))

    if n == 1:
        axes = np.expand_dims(axes, axis=0)

    for row, mask_path in enumerate(selected):
        image_path = IMAGE_DIR / f"{mask_path.stem}.jpg"

        if not image_path.exists():
            image_path = IMAGE_DIR / f"{mask_path.stem}.jpeg"

        if not image_path.exists():
            image_path = IMAGE_DIR / f"{mask_path.stem}.png"

        image = np.array(Image.open(image_path).convert("RGB"))
        mask = np.array(Image.open(mask_path))

        axes[row, 0].imshow(image)
        axes[row, 0].set_title(image_path.name)
        axes[row, 0].axis("off")

        axes[row, 1].imshow(
            create_overlay(image, mask)
        )
        axes[row, 1].set_title("Generated labels")
        axes[row, 1].axis("off")

    plt.tight_layout()
    plt.show()


# Important: review the masks before CNN training

These masks are automatically generated **pseudo-labels**.

Do not assume every mask is correct.

For an ISEF-quality dataset, inspect the generated masks and manually correct errors, especially:

- forehead boundaries
- cheek boundaries
- lower-lip region
- frames where the person turns their head
- frames with hair covering the forehead
- strong shadows or unusual lighting

After correction, the final dataset should look like:

```text
rppg_dataset/
├── images/
│   ├── frame_00001.jpg
│   ├── frame_00002.jpg
│   └── ...
└── masks/
    ├── frame_00001.png
    ├── frame_00002.png
    └── ...
```

Then run `02_train_segmentation_model.ipynb`.


# Optional: create masks from a video

The current notebook processes images.

For your eventual rPPG dataset, the better workflow is:

```text
video.mp4
   ↓
sample frames
   ↓
generate facial-region masks
   ↓
review/correct a subset
   ↓
train CNN
```

Do not train on every video frame from the same participant without considering participant-level train/test splitting. Frames from one person's video are highly correlated.
